# LABORATORIO DIRIGIDO N.° 03
**Semana 3 · SQL Avanzado — Joins y Agrupaciones**  
Clientes, pedidos y productos de TecnoAndina Retail con JOIN, agrupaciones y vistas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: Pilar Rocío Sayán Mejía  
Datos: `LD3_TecnoAndina_Retail.xlsx` (GitHub: Rociosayan/Repo_FDG_2026-2, carpeta semana03/datos/)

## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

- clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

- productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

- pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

- detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.

Las tablas se relacionan por id_cliente (clientes–pedidos), id_pedido (pedidos–detalles_pedido) e id_producto (productos–detalles_pedido).

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la celda 1: importa sqlite3 y pandas, crea la base en memoria, carga las cuatro hojas del Excel desde GitHub y cuenta los registros. Cada resultado se muestra con pd.read_sql_query; pandas muestra los NULL como NaN o None. Si reinicia el entorno, vuelva a ejecutar la celda 1.

**Celda 1 de 12**

In [ ]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)

## A. Relacionar tablas

### Ejercicio 1 — INNER JOIN y LEFT JOIN: clientes con y sin pedidos

El área comercial quiere saber cuántos clientes compraron y quiénes todavía no registran pedidos. INNER JOIN conserva solo las filas con pareja; LEFT JOIN conserva todas las filas de la tabla izquierda y completa con NULL.

Se solicita. Ejecute las celdas 2 y 3.

**Celda 2 de 12**

In [ ]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Celda 3 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 1.** ¿Cuántos pedidos y cuántos clientes distintos devuelve la celda 2? ¿Por qué INNER JOIN no muestra a todos los clientes registrados?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 2.** En la celda 3, ¿qué tabla es la izquierda y por qué se filtra con p.id_pedido IS NULL y no con = NULL? ¿Cuántos clientes no tienen pedidos?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 2 — RIGHT JOIN y FULL OUTER JOIN

Compras pregunta qué productos no se vendieron y la jefatura quiere comparar a los clientes entre agosto y septiembre. RIGHT JOIN conserva la tabla derecha; FULL OUTER JOIN conserva las filas de ambas tablas. Ambos requieren SQLite 3.39.0 o posterior.

Se solicita. Ejecute la celda 4 (versión y RIGHT JOIN) y la celda 5, que crea un resumen por cliente para cada mes y los une con FULL OUTER JOIN.

**Celda 4 de 12**

In [ ]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))

**Celda 5 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 3.** ¿Qué versión de SQLite informó la celda 4? En el RIGHT JOIN, ¿qué tabla es la derecha y qué productos no se vendieron?

**Respuesta:** _Escriba aquí su respuesta._

**Pregunta 4.** En la celda 5, ¿cuántos clientes compraron solo en agosto, solo en septiembre y en ambos meses? ¿Cómo los reconoce por los NULL?

**Respuesta:** _Escriba aquí su respuesta._

## B. Agrupar y filtrar grupos

### Ejercicio 3 — WHERE, GROUP BY y HAVING

La jefatura pide los clientes con al menos dos pedidos en septiembre. WHERE filtra filas antes de agrupar; GROUP BY forma un grupo por cliente; HAVING filtra los grupos ya formados.

Se solicita. Ejecute la celda 6.

**Celda 6 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 5.** ¿Qué filtra WHERE y qué filtra HAVING? Un cliente con tres pedidos en total, pero uno solo en septiembre, ¿aparece? Justifique.

**Respuesta:** _Escriba aquí su respuesta._

## C. Consultar y reutilizar resultados

### Ejercicio 4 — Subconsulta: pedidos sobre el promedio por pedido

Logística quiere revisar los pedidos que superan el monto de un pedido promedio. La subconsulta calcula primero el promedio; la consulta exterior compara cada pedido con ese valor.

Se solicita. Ejecute la celda 7.

**Celda 7 de 12**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 7.** ¿Qué devuelve la subconsulta y cuántos pedidos la superan? ¿Qué clientes aparecen más de una vez?

**Respuesta:** _Escriba aquí su respuesta._

### Ejercicio 5 — Vista: guardar el resumen por cliente

Varias áreas piden el mismo resumen por cliente. Una vista guarda la consulta con un nombre y se vuelve a ejecutar cada vez que se consulta.

Se solicita. Ejecute la celda 8, que crea vista_resumen_cliente y la consulta.

**Celda 8 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta 8.** ¿Qué guarda la vista: los datos o la consulta? ¿Qué columna distingue a los dos clientes llamados Carlos Mendoza?

**Respuesta:** _Escriba aquí su respuesta._

## Actividad 3 — Informe analítico comparado

La gerencia evalúa ofrecer beneficios a los clientes que más compraron. Ejecute la celda 9, complete la tabla y responda citando cifras. Liderar por monto en dos meses no convierte a un cliente en el más rentable.

**Celda 9 de 12**

In [ ]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

| Posición | id_cliente | Nombre | Ciudad | Pedidos | Monto acumulado (S/) | Promedio por pedido (S/) |
|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |
| 3 |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |

**A.** ¿Qué clientes lideran por monto acumulado y quién tiene el mayor promedio por pedido?

**Respuesta:** _Escriba aquí su respuesta._

**B.** ¿Qué información adicional pediría antes de recomendar beneficios? Redacte una recomendación prudente en tres líneas.

**Respuesta:** _Escriba aquí su respuesta._

## Reto de aplicación y retroalimentación

Indicaciones. Cada reto presenta la consulta ya escrita: ejecútela, identifique qué hace cada cláusula e interprete el resultado en no más de cuatro líneas. Los retos son la base de la actividad calificada de la semana.

### Reto 1 — Clientes sin pedidos por ciudad

Marketing quiere saber en qué ciudades se concentran los clientes que todavía no tienen pedidos. Un cliente sin pedidos no es necesariamente un cliente perdido.

**Celda 10 de 12**

In [ ]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

**Pregunta de cierre.** ¿Cuántos clientes sin pedidos hay y en qué ciudades se concentran? ¿Qué cláusula los conserva y cuál los aísla?

**Respuesta:** _Escriba aquí su respuesta._

### Reto 2 — Ingresos por categoría con un umbral

Finanzas quiere saber qué categorías generaron al menos S/ 10 000 entre agosto y septiembre. El ingreso se calcula desde las líneas de detalle.

**Celda 11 de 12**

In [ ]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

**Pregunta de cierre.** ¿Qué categorías superan el umbral y cuál queda fuera? ¿Por qué la condición va en HAVING y no en WHERE?

**Respuesta:** _Escriba aquí su respuesta._

### Reto 3 — Clientes sobre el promedio por cliente

Comercial quiere identificar a los clientes cuyo monto acumulado supera el promedio de los clientes que compraron. La consulta usa la vista de la celda 8 y una subconsulta.

**Celda 12 de 12**

In [ ]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

**Pregunta de cierre.** ¿Qué valor devuelve la subconsulta y cuántos clientes lo superan? ¿Por qué no debe compararse con el promedio por pedido del ejercicio 4?

**Respuesta:** _Escriba aquí su respuesta._

## Conclusiones

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

1. _Escriba aquí su conclusión._

2. _Escriba aquí su conclusión._

3. _Escriba aquí su conclusión._